# FNSPID + FinBERT: does real headline sentiment beat the Alpha Vantage dead end?

**Run this on Kaggle** (Settings → Accelerator: GPU T4, Internet: On). It streams the two FNSPID news CSVs directly from Hugging Face over HTTP, filtering to our 21-ticker universe as it goes — it never writes the full 23GB file to disk, so this works regardless of Kaggle's session disk quota.

**Why this exists:** `market-impact-predictor`'s Alpha Vantage-based pipeline tried three feature configs (raw sentiment scores, +ticker, +topic categories) across two dataset sizes and never beat a trivial majority-class baseline. The diagnosis was that AV's 3 proprietary sentiment numbers just don't carry next-few-days direction signal — not a data volume problem. This notebook tests a different hypothesis: does *our own* FinBERT sentiment score, computed directly on real headlines from FNSPID (2019-2020 Benzinga coverage for these tickers), do any better?

**Known limitation, stated up front:** FNSPID's news CSVs have an `Article` (full body) column, but it's empty for every row we checked — confirmed via a direct byte-range fetch, not assumed. So this is headline-level sentiment, not full-article sentiment. Still a real upgrade over AV's opaque score (it's ours, it's explainable), just not the richest possible version of this idea.

**Outputs** (in `/kaggle/working/`, download from the notebook's Output tab when done):
- `fnspid_dataset.csv` — the processed, labeled dataset (small — just our 21 tickers)
- `direction_classifier_fnspid.joblib`, `magnitude_regressor_fnspid.joblib` — trained models

Copy those into the local project's `data/processed/` and `models/` directories afterward.

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import io
import zipfile
from datetime import datetime, time as dtime, timezone

import joblib
import numpy as np
import pandas as pd
import requests
import torch
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
    precision_score, recall_score, root_mean_squared_error,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from transformers import pipeline as hf_pipeline

# Same 21 tickers as the local project's src/ingest/tickers.py - copied
# verbatim so the trained model matches the existing pipeline's feature
# space (ticker is a one-hot feature downstream).
TICKER_UNIVERSE = [
    "AAPL", "MSFT", "NVDA", "GOOGL",   # tech
    "JPM", "BAC", "V",                  # financials
    "JNJ", "UNH", "PFE",                # healthcare
    "AMZN", "WMT", "MCD",               # consumer
    "XOM", "CVX",                       # energy
    "BA", "CAT",                        # industrials
    "META", "DIS",                      # communication
    "TSLA", "GM",                       # auto
]

# Same forward-return horizon as src/features/build_dataset.py, so this
# model is directly comparable to the AV-based one.
HORIZON_DAYS = 3
MARKET_CLOSE_UTC_HOUR = 20  # approximate US market close, same simplification as the local project

device = 0 if torch.cuda.is_available() else -1
print("GPU available:" , torch.cuda.is_available())

## Step 1 — stream-filter the news CSV to our 21 tickers

`All_external.csv` (5.7GB) is tried first since it's smaller and faster to stream. If coverage for our tickers looks thin, the larger `nasdaq_exteral_data.csv` (23GB, sourced from more publishers per the FNSPID paper) is scanned too — that pass is slower (network-bound streaming of 23GB), so it only runs if actually needed.

In [ ]:
NEWS_URLS = {
    "small": "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/All_external.csv",
    "large": "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/nasdaq_exteral_data.csv",
}

TICKER_SET = set(TICKER_UNIVERSE)


def stream_filter_news(url: str, chunksize: int = 200_000) -> pd.DataFrame:
    """Streams the CSV over HTTP in chunks, keeping only rows for our
    tickers with a non-empty headline - never materializes the full file
    on disk or in memory."""
    keep = []
    usecols = ["Date", "Article_title", "Stock_symbol", "Publisher"]
    total_rows = 0
    for i, chunk in enumerate(pd.read_csv(url, usecols=usecols, chunksize=chunksize, on_bad_lines="skip")):
        total_rows += len(chunk)
        matched = chunk[chunk["Stock_symbol"].isin(TICKER_SET) & chunk["Article_title"].notna()]
        if len(matched):
            keep.append(matched)
        if i % 20 == 0:
            found = sum(len(k) for k in keep)
            print(f"  scanned {total_rows:,} rows so far, {found:,} matched our tickers")
    return pd.concat(keep, ignore_index=True) if keep else pd.DataFrame(columns=usecols)


print("Scanning All_external.csv (5.7GB, ~20-40 min depending on Kaggle's network)...")
news_df = stream_filter_news(NEWS_URLS["small"])
print(f"\nMatched {len(news_df):,} headlines across our 21 tickers from the small file.")
print(news_df["Stock_symbol"].value_counts())

In [ ]:
# Only scan the big 23GB file if the small one left any ticker thin -
# this pass is much slower, so it's conditional, not automatic.
MIN_PER_TICKER = 200
counts = news_df["Stock_symbol"].value_counts()
thin_tickers = [t for t in TICKER_UNIVERSE if counts.get(t, 0) < MIN_PER_TICKER]

if thin_tickers:
    print(f"Thin coverage for {thin_tickers} - scanning the large file too (this is slow, be patient)...")
    large_df = stream_filter_news(NEWS_URLS["large"])
    news_df = pd.concat([news_df, large_df], ignore_index=True)
    news_df = news_df.drop_duplicates(subset=["Date", "Article_title", "Stock_symbol"])
    print(f"Combined total after de-dup: {len(news_df):,} headlines.")
else:
    print("Coverage looks sufficient from the small file alone - skipping the 23GB scan.")

news_df.to_csv("/kaggle/working/filtered_headlines_raw.csv", index=False)
print(news_df["Stock_symbol"].value_counts())

## Step 2 — price data

`full_history.zip` is only 589MB, so it's downloaded in full, then we extract just the 21 member files we need. **This assumes one CSV per ticker inside the zip (the common convention for this kind of dump) — the cell below prints the actual zip contents first so you can see immediately if that assumption is wrong and adjust the filename pattern.**

In [ ]:
PRICE_URL = "https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_price/full_history.zip"

resp = requests.get(PRICE_URL)
resp.raise_for_status()
zf = zipfile.ZipFile(io.BytesIO(resp.content))

names = zf.namelist()
print(f"{len(names)} files in the zip. First 10:")
print(names[:10])

In [ ]:
# Adjust this if the printed listing above doesn't match "<TICKER>.csv"
# (e.g. it might be nested under a folder, or lowercase).
def find_member(ticker: str) -> str | None:
    candidates = [n for n in names if n.rsplit("/", 1)[-1].upper() in (f"{ticker}.CSV",)]
    return candidates[0] if candidates else None

prices = {}
missing = []
for ticker in TICKER_UNIVERSE:
    member = find_member(ticker)
    if member is None:
        missing.append(ticker)
        continue
    with zf.open(member) as f:
        df = pd.read_csv(f)
    df.columns = [c.strip() for c in df.columns]
    date_col = "Date" if "Date" in df.columns else df.columns[0]
    close_col = "Adj Close" if "Adj Close" in df.columns else "Close"
    df = df[[date_col, close_col]].rename(columns={date_col: "date", close_col: "close"})
    df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
    df = df.dropna().sort_values("date").reset_index(drop=True)
    prices[ticker] = df

print(f"Loaded price history for {len(prices)}/{len(TICKER_UNIVERSE)} tickers.")
if missing:
    print(f"MISSING (check the naming convention above and fix find_member): {missing}")

## Step 3 — labels

Same logic as `src/features/labels.py` in the local project, copied verbatim: an article published during market hours reacts in that day's close; one published after close (or on a non-trading day) can't be reacted to until the next session.

In [ ]:
def compute_label_from_series(dates, closes, published_at, horizon_days):
    article_date = published_at.date().isoformat()
    cutoff = datetime.combine(published_at.date(), dtime(MARKET_CLOSE_UTC_HOUR, 0), tzinfo=timezone.utc)

    if article_date in dates and published_at <= cutoff:
        ref_idx = dates.index(article_date)
    else:
        ref_idx = next((i for i, d in enumerate(dates) if d > article_date), None)
        if ref_idx is None:
            return None

    target_idx = ref_idx + horizon_days
    if target_idx >= len(dates):
        return None

    reference_close = closes[ref_idx]
    target_close = closes[target_idx]
    return {
        "reference_date": dates[ref_idx],
        "target_date": dates[target_idx],
        "direction": 1 if target_close > reference_close else 0,
        "return_pct": (target_close - reference_close) / reference_close,
    }


def parse_fnspid_date(raw: str) -> datetime | None:
    # FNSPID's Date column has shown up as e.g. "2020-06-05 06:30:54 UTC"
    try:
        return pd.to_datetime(raw, utc=True).to_pydatetime()
    except (ValueError, TypeError):
        return None


labeled_rows = []
skipped = 0
for ticker, group in news_df.groupby("Stock_symbol"):
    if ticker not in prices:
        continue
    price_df = prices[ticker]
    dates = price_df["date"].tolist()
    closes = price_df["close"].tolist()
    for _, row in group.iterrows():
        published_at = parse_fnspid_date(row["Date"])
        if published_at is None:
            skipped += 1
            continue
        label = compute_label_from_series(dates, closes, published_at, HORIZON_DAYS)
        if label is None:
            skipped += 1
            continue
        labeled_rows.append({
            "ticker": ticker,
            "headline": row["Article_title"],
            "time_published": row["Date"],
            **label,
        })

labeled_df = pd.DataFrame(labeled_rows)
print(f"Built {len(labeled_df):,} labeled examples ({skipped:,} skipped - not enough price history around them).")

## Step 4 — FinBERT sentiment on headlines

`ProsusAI/finbert` is a BERT model fine-tuned specifically for financial sentiment (positive/negative/neutral). Score = P(positive) − P(negative), a continuous value in [-1, 1] we compute ourselves, not trusted from a third party.

In [ ]:
sentiment_pipe = hf_pipeline(
    "text-classification", model="ProsusAI/finbert", top_k=None, device=device, batch_size=64, truncation=True,
)

headlines = labeled_df["headline"].tolist()
scores = []
BATCH = 256
for i in range(0, len(headlines), BATCH):
    batch = headlines[i:i + BATCH]
    results = sentiment_pipe(batch)
    for r in results:
        by_label = {d["label"].lower(): d["score"] for d in r}
        scores.append(by_label.get("positive", 0.0) - by_label.get("negative", 0.0))
    if i % (BATCH * 20) == 0:
        print(f"  scored {i + len(batch):,}/{len(headlines):,} headlines")

labeled_df["finbert_sentiment"] = scores
print("Done. Sentiment score distribution:")
print(labeled_df["finbert_sentiment"].describe())

## Step 5 — assemble dataset, time-based split

Split by time, not randomly - same reasoning as the local project's `time_based_split`: a random split would let the model implicitly train on articles published after some test examples, which is lookahead leakage.

In [ ]:
MIN_EXAMPLES_FOR_SPLIT = 30

labeled_df["time_published_dt"] = pd.to_datetime(labeled_df["time_published"], utc=True)
labeled_df = labeled_df.sort_values("time_published_dt").reset_index(drop=True)

if len(labeled_df) < MIN_EXAMPLES_FOR_SPLIT:
    raise SystemExit(f"Only {len(labeled_df)} examples - below the {MIN_EXAMPLES_FOR_SPLIT} minimum. Not enough to evaluate honestly.")

split_idx = int(len(labeled_df) * 0.8)
train_df, test_df = labeled_df.iloc[:split_idx].copy(), labeled_df.iloc[split_idx:].copy()
print(f"train={len(train_df)}  test={len(test_df)}")

labeled_df.to_csv("/kaggle/working/fnspid_dataset.csv", index=False)

## Step 6 — train & honest-baseline comparison

Same pattern as `train_improved.py` in the local project: always compare against a trivial majority-class baseline, and warn explicitly if nothing beats it. This is the actual test of whether FinBERT-on-headlines does better than Alpha Vantage's scores did.

In [ ]:
NUMERIC_FEATURES = ["finbert_sentiment"]
CATEGORICAL_FEATURES = ["ticker"]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_FEATURES),
    ])

trivial = DummyClassifier(strategy="most_frequent")
trivial.fit(train_df[ALL_FEATURES], train_df["direction"])
trivial_pred = trivial.predict(test_df[ALL_FEATURES])
trivial_acc = accuracy_score(test_df["direction"], trivial_pred)
trivial_f1 = f1_score(test_df["direction"], trivial_pred, zero_division=0)
print(f"Trivial majority-class baseline: accuracy={trivial_acc:.3f}  f1={trivial_f1:.3f}\n")

configs = {
    "Logistic + scaling + ticker": Pipeline([("prep", make_preprocessor()), ("clf", LogisticRegression())]),
    "Logistic + balanced": Pipeline([("prep", make_preprocessor()), ("clf", LogisticRegression(class_weight="balanced"))]),
    "HistGradientBoosting": Pipeline([("prep", make_preprocessor()), ("clf", HistGradientBoostingClassifier())]),
}

results = []
for name, model in configs.items():
    model.fit(train_df[ALL_FEATURES], train_df["direction"])
    pred = model.predict(test_df[ALL_FEATURES])
    acc = accuracy_score(test_df["direction"], pred)
    f1 = f1_score(test_df["direction"], pred, zero_division=0)
    print(f"{name}:")
    print(f"  accuracy={acc:.3f}  precision={precision_score(test_df['direction'], pred, zero_division=0):.3f} "
          f"recall={recall_score(test_df['direction'], pred, zero_division=0):.3f}  f1={f1:.3f}")
    print(f"  confusion matrix [[TN FP][FN TP]]:\n{confusion_matrix(test_df['direction'], pred)}\n")
    results.append((name, model, f1))

best_name, best_model, best_f1 = max(results, key=lambda r: r[2])
print(f"Best by F1: {best_name} (f1={best_f1:.3f})")
if best_f1 <= trivial_f1:
    print(f"WARNING: best model (f1={best_f1:.3f}) does not beat the trivial baseline (f1={trivial_f1:.3f}). "
          f"FinBERT headline sentiment does not fix the signal problem either - report this honestly.")
else:
    print(f"This DOES beat the trivial baseline (f1={trivial_f1:.3f}) - a real result, worth digging into further.")

In [ ]:
# Magnitude regressor, same comparison shape as the local project.
mag_configs = {
    "Linear": Pipeline([("prep", make_preprocessor()), ("reg", LinearRegression())]),
    "HistGradientBoosting": Pipeline([("prep", make_preprocessor()), ("reg", HistGradientBoostingRegressor())]),
}
mag_results = []
for name, model in mag_configs.items():
    model.fit(train_df[ALL_FEATURES], train_df["return_pct"])
    pred = model.predict(test_df[ALL_FEATURES])
    mae = mean_absolute_error(test_df["return_pct"], pred)
    rmse = root_mean_squared_error(test_df["return_pct"], pred)
    print(f"{name}: MAE={mae:.4f}  RMSE={rmse:.4f}")
    mag_results.append((name, model, mae))

best_mag_name, best_mag_model, best_mae = min(mag_results, key=lambda r: r[2])
print(f"\nBest by MAE: {best_mag_name} (mae={best_mae:.4f})")

## Step 7 — save outputs

Download these from the notebook's Output tab and copy into the local project: `fnspid_dataset.csv` → `data/processed/`, the two `.joblib` files → `models/`.

In [ ]:
joblib.dump(best_model, "/kaggle/working/direction_classifier_fnspid.joblib")
joblib.dump(best_mag_model, "/kaggle/working/magnitude_regressor_fnspid.joblib")
print("Saved. Check the Output tab for: fnspid_dataset.csv, direction_classifier_fnspid.joblib, magnitude_regressor_fnspid.joblib")